# Data Preprocessing

In this notebook, I will prepare the dataset for use in a model by handling data type issues, encoding categorical variables, and ensuring consistent transformations between training and data sets. I will start by reading in the training and testing data that was split during EDA.


In [27]:
import pandas as pd

X_train = pd.read_parquet("../data/X_train.parquet")
X_test = pd.read_parquet("../data/X_test.parquet")

y_train = pd.read_parquet("../data/y_train.parquet").squeeze()
y_test = pd.read_parquet("../data/y_test.parquet").squeeze()

## Handle Data Type Issues and Missing Values

We will start with dealing with any data type issues and missing values in the data set. To handle missing numeric values, I will fill them using the median, and for missing categorical values I will the most frequent value.

In [28]:
from sklearn.impute import SimpleImputer

X_train.drop(columns=["customerID"], inplace=True)
X_test.drop(columns=["customerID"], inplace=True)

X_train["TotalCharges"] = pd.to_numeric(X_train["TotalCharges"], errors="coerce")
X_test["TotalCharges"] = pd.to_numeric(X_test["TotalCharges"], errors="coerce")

num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include="object").columns

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

num_imputer.fit(X_train[num_cols])
cat_imputer.fit(X_train[cat_cols])

X_train_num = num_imputer.transform(X_train[num_cols])
X_test_num = num_imputer.transform(X_test[num_cols])

X_train_cat = cat_imputer.transform(X_train[cat_cols])
X_test_cat = cat_imputer.transform(X_test[cat_cols])

## Encode categorical variables

Right now all of the categorical variables are just stored as text, so they need to be encoded. Once they are encoded we can combine them back with the numeric features

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat_encoded = encoder.fit_transform(X_train_cat)
X_test_cat_encoded = encoder.transform(X_test_cat)

X_train_final = np.hstack([X_train_num, X_train_cat_encoded])
X_test_final = np.hstack([X_test_num, X_test_cat_encoded])

y_train_final = y_train["Churn"].map({"Yes": 1, "No": 0})
y_test_final = y_test["Churn"].map({"Yes": 1, "No": 0})


## Save the data

In [ ]:
X_train_final = pd.DataFrame(X_train_final)
X_test_final = pd.DataFrame(X_test_final)

X_train_final.to_parquet("../data/X_train_final.parquet", index=False)
X_test_final.to_parquet("../data/X_test_final.parquet", index=False)

y_train_final = pd.DataFrame(y_train_final, columns=["Churn"])
y_test_final = pd.DataFrame(y_test_final, columns=["Churn"])

y_train_final.to_parquet("../data/y_train_final.parquet", index=False)
y_test_final.to_parquet("../data/y_test_final.parquet", index=False)